In [ ]:
# ══════════════════════════════════════════════════════════════
# M3_F02 — LOGISTICS  |  Cellule 1 : Montage Drive + Install
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!pip install -q flask flask-cors

import os, sys
from pathlib import Path

CODEBASE = Path("/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F02_LOGISTICS/CODEBASE")
sys.path.insert(0, str(CODEBASE))
print("Drive monté — CODEBASE :", CODEBASE)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 2 : Vérification Inputs
# ══════════════════════════════════════════════════════════════
DRIVE_ROOT  = Path("/content/drive/MyDrive/EXODUS_V3/M3")
AVATAR_PATH = DRIVE_ROOT / "SHARED" / "avatar.glb"
PROPS_DIR   = DRIVE_ROOT / "SHARED" / "props"
F01_REPORT  = DRIVE_ROOT / "F01_VALIDATION" / "OUT_REPORT" / "m3_f01_report.json"

import json

# F01 report
if F01_REPORT.exists():
    with open(F01_REPORT) as f:
        f01 = json.load(f)
    print(f"F01 report     OK — clip: {f01.get("selected_clip", "?")} / has_audio: {f01.get("has_audio")}")
else:
    print("F01 report     ABSENT — lancer F01 d'abord")

# Avatar
size = AVATAR_PATH.stat().st_size if AVATAR_PATH.exists() else 0
print(f"avatar.glb     {"OK  (" + str(size//1024) + " KB)" if AVATAR_PATH.exists() else "ABSENT"}")

# Props
props = list(PROPS_DIR.glob("*.glb")) if PROPS_DIR.exists() else []
print(f"props/         {len(props)} fichier(s) GLB disponible(s)")
for p in props:
    print(f"  - {p.name}  ({p.stat().st_size//1024} KB)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 3 : Lancement Flask
# ══════════════════════════════════════════════════════════════
import shutil, threading, time
from google.colab.output import eval_js

LOCAL = Path("/content/m3_f02")
LOCAL.mkdir(exist_ok=True)
for f in CODEBASE.glob("*"):
    shutil.copy(f, LOCAL / f.name)

PORT = 5002

def run_flask():
    os.chdir(str(LOCAL))
    os.system(f"python m3_f02_flask.py")

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print(f"
Viewer F02 disponible :
{url}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 4 : Mode SKIP headless (optionnel)
# Utiliser si pas de props à attacher
# ══════════════════════════════════════════════════════════════
# import json
# from pathlib import Path
#
# DRIVE_ROOT  = Path("/content/drive/MyDrive/EXODUS_V3/M3")
# OUT_DIR     = DRIVE_ROOT / "F02_LOGISTICS" / "OUT"
# OUT_DIR.mkdir(parents=True, exist_ok=True)
#
# report = {"status": "SKIPPED", "has_props": False}
# with open(OUT_DIR / "m3_f02_report.json", "w") as f:
#     json.dump(report, f, indent=2)
# print("Rapport SKIPPED sauvé :", OUT_DIR / "m3_f02_report.json")